# Logical Layer: Apps, Connections & Converters

The logical layer models **end-to-end communication** on top of the physical network. In this notebook you will:

- Create apps on devices
- Define connections between apps with protocols and information items
- Understand the `peers` / `producer` data-flow model
- Model protocol converters with chained connections
- Check whether a connection is topologically feasible
- Visualize the logical layer graph

## Setup — recreate the physical network from notebook 02

In [ ]:
import pandaict as pi
from pandaict.utils.enums import NodeType, DeviceType, AppType, InfoItem
from pandaict.plot import create_plot
import matplotlib.pyplot as plt

net = pi.create_empty_network(name='Logical Layer Demo')

# Physical layer (same as notebook 02, condensed)
n_core  = pi.create_ict_node(net, name='Core-Router',   node_type=NodeType.ROUTER, link_types=['Fiber','LAN1000'])
n_sw1   = pi.create_ict_node(net, name='Switch-1',      node_type=NodeType.SWITCH, link_types=['LAN1000'])
n_sw2   = pi.create_ict_node(net, name='Switch-2',      node_type=NodeType.SWITCH, link_types=['LAN1000'])
n_field = pi.create_ict_node(net, name='Field-Switch',   node_type=NodeType.SWITCH, link_types=['LAN1000'])

ctrl_server = pi.create_ict_device(net, name='CTRL-Server', device_type=DeviceType.SERVER)
rtu_a       = pi.create_ict_device(net, name='RTU-A',       device_type=DeviceType.RTU)
rtu_b       = pi.create_ict_device(net, name='RTU-B',       device_type=DeviceType.RTU)
(mon_dev, mon_app) = pi.create_ict_devapp(net, name='Monitoring',
                                          device_type=DeviceType.SERVER,
                                          app_type=AppType.WEBSERVER)

pi.create_ict_link(net, name='L-SW1-Core',   from_elt=n_sw1,        to_elt=[n_core],       link_type='LAN1000')
pi.create_ict_link(net, name='L-Core-SW2',   from_elt=n_core,       to_elt=[n_sw2],        link_type='LAN1000')
pi.create_ict_link(net, name='L-SW2-Field',  from_elt=n_sw2,        to_elt=[n_field],      link_type='LAN1000')
pi.create_ict_link(net, name='L-Field-SW1',  from_elt=n_field,      to_elt=[n_sw1],        link_type='LAN1000')
pi.create_ict_link(net, name='L-Core-CTRL',  from_elt=n_core,       to_elt=[ctrl_server],  link_type='LAN1000')
pi.create_ict_link(net, name='L-Field-RTU-A',from_elt=n_field,      to_elt=[rtu_a],        link_type='LAN1000')
pi.create_ict_link(net, name='L-Field-RTU-B',from_elt=n_field,      to_elt=[rtu_b],        link_type='LAN1000')
pi.create_ict_link(net, name='L-SW2-Mon',    from_elt=n_sw2,        to_elt=[mon_dev],      link_type='LAN1000')

print('Physical layer ready:', len(net.node), 'nodes,', len(net.device), 'devices,', len(net.link), 'links')

## Apps

An **app** is a service running on a device. Each app belongs to exactly one device. Built-in types include:

- `AppType.CLIENT` — initiates requests
- `AppType.WEBSERVER` — serves responses
- `AppType.CONVERTER` — translates between two protocols (requires DeviceType.CONVERTER)
- `AppType.FILESERVER`, `AppType.RESTSERVER`, `AppType.RESTCLIENT`

Key parameters: `name`, `app_type`, `device` (the device tuple it runs on).

In [ ]:
# 1. Create a standard type for the RS485 protocol (see notebook 06 for details)
pi.create_std_type(net, {'info_items': [InfoItem.REQUEST, InfoItem.VALUE]},
                   name='MODBUS-RS485', element='protocol')

# 2. Create a converter device and app
(conv_dev, conv_app) = pi.create_ict_devapp(net, name='Proto-Converter',
                                            device_type=DeviceType.CONVERTER,
                                            app_type=AppType.CONVERTER,
                                            conversions=[('TCP', 'MODBUS-RS485', 0.001, 0.001)])

# 3. Cable the converter between Core-Router and RTU-B
pi.create_ict_link(net, name='L-Core-Conv',  from_elt=n_core,  to_elt=[conv_dev], link_type='LAN1000')
pi.create_ict_link(net, name='L-Conv-RTUB', from_elt=conv_dev, to_elt=[rtu_b],    link_type='LAN1000')

# 4. Create CHAINED connections through the converter
#    CTRL --> Converter (TCP)     +     Converter --> RTU-B (MODBUS-RS485)
conn_ctrl_conv = pi.create_ict_connection(net, name='CTRL-to-Converter',
    from_app=app_ctrl, to_app=conv_app, protocol='TCP',
    info_items=[InfoItem.REQUEST, InfoItem.VALUE],
    peers=(app_ctrl, app_rtub), producer=[0, 1])

conn_conv_rtub = pi.create_ict_connection(net, name='Converter-to-RTUB',
    from_app=conv_app, to_app=app_rtub, protocol='MODBUS-RS485',
    info_items=[InfoItem.REQUEST, InfoItem.VALUE],
    peers=(app_ctrl, app_rtub), producer=[0, 1])

print(f'Chained connections: {conn_ctrl_conv} --> {conn_conv_rtub}')
print(f'Protocols: {net.conn.iloc[conn_ctrl_conv[1]]["protocol"]} --> {net.conn.iloc[conn_conv_rtub[1]]["protocol"]}')


# View the converter's conversions and the chained connections
print("Converter conversions:")
display(net.app[net.app['type'] == AppType.CONVERTER][['name', 'type', 'device', 'conversions']])

print("\nAll connections:")
display(net.conn[['name', 'protocol', 'from', 'to']])


### How chained connections work

The converter splits one end-to-end communication into two connections with different protocols:

1. **CTRL-to-Converter** (TCP): the control center app sends a TCP request to the converter app
2. The **converter app** translates TCP to MODBUS-RS485 (defined in its `conversions` list)
3. **Converter-to-RTUB** (MODBUS-RS485): the translated message reaches RTU-B over the RS485 protocol

Both connections share the same `peers` (app_ctrl and app_rtub) and `producer` configuration,
meaning the end-to-end data flow is preserved across the protocol boundary. The `peers` field
tracks who the *real* communicating parties are, not the converter intermediate.


In [ ]:
from pandaict.topology.routing import is_connection_possible

print('RTU-A reachable (direct):', is_connection_possible(net, conn_rtua))
print('RTU-B reachable (via converter):', is_connection_possible(net, conn_ctrl_conv), '|', is_connection_possible(net, conn_conv_rtub))


**Understanding `peers` and `producer`**:
- `peers = (app_ctrl, app_rtua)` — the two apps involved
- `info_items = [REQUEST, VALUE]` — what gets transmitted
- `producer = [0, 1]` means:
  - `peers[0]` (which is `app_ctrl`) produces `REQUEST`
  - `peers[1]` (which is `app_rtua`) produces `VALUE`

This information is essential for routing: the system must know which direction each data item flows.

### CTRL server also requests from RTU-B

In [ ]:
conn_rtub = pi.create_ict_connection(net, name='CTRL-to-RTU-B',
                                     from_app=app_ctrl, to_app=app_rtub,
                                     protocol='TCP',
                                     info_items=[InfoItem.REQUEST, InfoItem.VALUE],
                                     peers=(app_ctrl, app_rtub),
                                     producer=[0, 1])

## Checking connectivity

`is_connection_possible()` checks whether a path exists through the physical layer for a given connection:

In [ ]:
from pandaict.topology.routing import is_connection_possible

print('RTU-A reachable:', is_connection_possible(net, conn_rtua))
print('RTU-B reachable:', is_connection_possible(net, conn_rtub))

## Protocol converters

A converter translates between two protocols (e.g., MODBUS-TCP → MODBUS-RS485). In pandaict this is modeled as:
- A device of type `DeviceType.CONVERTER`
- An app of type `AppType.CONVERTER` with a `conversions` list
- A pair of **chained connections**: one on each side of the converter

In [ ]:
# 1. Create a standard type for the RS485 protocol (see notebook 06 for details)
pi.create_std_type(net, {'info_items': [InfoItem.REQUEST, InfoItem.VALUE]},
                   name='MODBUS-RS485', element='protocol')

# 2. Create a converter device and app
(conv_dev, conv_app) = pi.create_ict_devapp(net, name='Proto-Converter',
                                            device_type=DeviceType.CONVERTER,
                                            app_type=AppType.CONVERTER,
                                            conversions=[('TCP', 'MODBUS-RS485', 0.001, 0.001)])

# 3. Place the converter on a network path
pi.create_ict_link(net, name='L-Core-Conv', from_elt=n_core, to_elt=[conv_dev], link_type='LAN1000')
pi.create_ict_link(net, name='L-Conv-RTUA', from_elt=conv_dev, to_elt=[rtu_a], link_type='LAN1000')

print('Converter ready. Apps:', len(net.app), '| Devices:', len(net.device))

In [ ]:
# View the converter's conversions
display(net.app[net.app['type'] == AppType.CONVERTER][['name', 'type', 'device', 'conversions']])

## Visualizing the logical layer

`create_plot(draw_type='app')` shows apps as nodes and connections as edges:

In [ ]:
create_plot(net, draw_type='app')
plt.title('Logical Layer — Apps & Connections')
plt.tight_layout()
plt.show()

## Summary

You now understand how to:
- Create apps on devices and wire them with end-to-end connections
- Model data-flow direction using `peers` / `producer`
- Check connectivity with `is_connection_possible()`
- Model protocol conversion with converter devices/apps and chained connections

Next we run actual **calculations** on this network — routing paths, reliability metrics, BER, and data volumes.

Next: [04 – Workflow & Analysis](04_workflow_analysis.ipynb)